# 📦 Data Shipping 101: A Beginner's Guide to Apache Avro

Welcome back! In our previous lessons, we looked at Thrift and Protobuf. Now, let's look at **Apache Avro**, the most space-efficient way to pack data.

### The Big Idea: The Shipping Manifest
Imagine you are shipping a box of items. 
- **JSON** is like writing a description on every single item in the box. Very clear, but takes up too much room.
- **Thrift/Protobuf** is like putting a small ID tag (a "Menu Number") on every item. Much better!
- **Avro** is different. It puts **nothing** on the items. It just packs them in a specific order and attaches a **Manifest (the Schema)** to the outside of the box. 

To unpack an Avro box, you *must* have the manifest. Without it, the data is just a pile of mystery bytes.

## 1. Why use Avro?

1.  **Ultimate Compactness**: Because there are no labels or tags inside the data, Avro files are incredibly small.
2.  **Great for Big Data**: In systems like Hadoop or Spark, we store the "Manifest" at the top of the file once, and then store millions of rows of data below it. 
3.  **Schema Evolution**: Avro is very smart about changes. If the "Packer" uses one version of the manifest and the "Unpacker" uses a slightly newer one, Avro can automatically figure out how to bridge the gap.

## 2. Defining the Manifest (The Schema)

Avro can describe data using JSON. Here is what our `Person` looks like in Avro:

```json
{
  "type": "record",
  "name": "Person",
  "fields": [
    { "name": "userName", "type": "string" },
    { "name": "favoriteNumber", "type": ["null", "long"], "default": null },
    { "name": "interests", "type": { "type": "array", "items": "string" } }
  ]
}
```

Notice that there are **no numbers** (like `1:`, `2:`) here! Avro relies entirely on the **order** of the fields defined in this manifest.

## 3. Hands-on: Decoding a Real-World File

Instead of creating our own data, let's look at a real-world example. We have a file containing genomic data. We'll use the `fastavro` library to read it.

Think of this code as **Opening the Shipping Container** and **Reading the Manifest**.

In [1]:
import fastavro
import copy
import json
from pprint import pprint

In [2]:
with open('../data/1k.variants.avro', 'rb') as f:
    # The reader automatically looks for the manifest (schema) at the start of the file
    reader = fastavro.reader(f)
    
    # We pull the records out of the container
    genomic_var_1k = [sample for sample in reader]
    
    # We inspect the Manifest (Schema) that was used to pack this data
    metadata = copy.deepcopy(reader.metadata)
    writer_schema = copy.deepcopy(reader.writer_schema)
    schema_from_file = json.loads(metadata['avro.schema'])

In [3]:
# How many items were in the box?
print(f"Total records found: {len(genomic_var_1k)}")

Total records found: 1000


### Inspecting the Manifest
Let's see the rules that were used to pack this data. This tells us exactly what fields exist and what types they are.

In [4]:
pprint(schema_from_file)

{'fields': [{'doc': '* The variant ID.',
             'name': 'id',
             'type': ['null',
                      {'avro.java.string': 'String', 'type': 'string'}]},
            {'default': [],
             'doc': '* Other names used for this genomic variation.',
             'name': 'names',
             'type': {'items': {'avro.java.string': 'String', 'type': 'string'},
                      'type': 'array'}},
            {'doc': '* Chromosome where the genomic variation occurred.',
             'name': 'chromosome',
             'type': {'avro.java.string': 'String', 'type': 'string'}},
            {'doc': '* Normalized position where the genomic variation '
                    'starts.\n'
                    '         * <ul>\n'
                    '         * <li>SNVs have the same start and end '
                    'position</li>\n'
                    '         * <li>Insertions start in the last present '
                    'position: if the first nucleotide\n'
          

### Looking at the Unpacked Data
Now that we have the manifest, the computer can translate those mystery bytes back into something we can read.

In [5]:
print("First record in the file:")
pprint(genomic_var_1k[0])

First record in the file:
{'alternate': 'T',
 'annotation': {'additionalAttributes': None,
                'alternate': 'T',
                'ancestralAllele': None,
                'chromosome': '22',
                'consequenceTypes': [{'biotype': 'lincRNA',
                                      'cdnaPosition': 0,
                                      'cdsPosition': 0,
                                      'codon': '',
                                      'ensemblGeneId': 'ENSG00000233866',
                                      'ensemblTranscriptId': 'ENST00000424770',
                                      'geneName': 'LA16c-4G1.3',
                                      'proteinVariantAnnotation': {'alternate': '',
                                                                   'features': [],
                                                                   'functionalDescription': None,
                                                                   'keywords': [],
       

In [ ]:
print("All field names defined in the manifest:")
for f in schema_from_file["fields"]:
    print(f"- {f['name']}")

## (Optional) 🎓 Exercises

1. **Explore the Data**: The first record has many nested fields (fields inside fields). Can you print the `chromosome` and `start` position of the first sample?
2. **The Missing Manifest**: What would happen if you tried to read this file but provided a manifest that said every field was an integer instead of a string? (Hint: It would look like gibberish or crash!)
3. **Data Analysis**: Loop through the first 10 records and print out the `type` of each genomic variation.